In [2]:
# ============================================================
# ACTIVIDAD 11 | PIPELINE MODULAR DE DATOS CON PYSPARK
# PROYECTO INTEGRADOR: PyEconGDL (Fuente Piloto: Deuda Bruta)
# ============================================================
#
# OBJETIVO
# Diseñar, ejecutar y verificar un pipeline modular en PySpark
# utilizando como fuente piloto la serie histórica mensual de
# Deuda Bruta de la SHCP (2011 - 2026).
#
# CONTEXTO
# Esta ejecución valida la arquitectura modular del proyecto
# antes de incorporar el resto de los indicadores macroeconómicos
# (TIIE, INPC, IGAE e ingresos/gastos presupuestarios).
#
# CONCEPTOS APLICADOS
#
# 1. MODULARIDAD
#    Separación de responsabilidades entre extracción (src/ingestion),
#    calidad (src/quality), transformación (src/transformations)
#    y orquestación (pipeline.py).
#
# 2. CALIDAD DE DATOS
#    Validaciones de entrada (validate_input) y salida (validate_output)
#    para asegurar la integridad de los 186 registros mensuales.
#
# 3. PROCESAMIENTO
#    Construcción de la llave temporal PERIODO (a partir de CICLO y MES)
#    y comparación intermensual de la deuda bruta.
#
# 4. TRAZABILIDAD Y AUDITORÍA
#    Agrupación de trabajos en Spark UI (CLASE11_PIPELINE) y registro
#    automático de evidencia en reports/ultima_ejecucion.json.
#
# 5. PERSISTENCIA Y RELECTURA
#    Almacenamiento columnar en Parquet y comprobación posterior
#    mediante verify_load().
#
# FLUJO DEL PIPELINE
# Extract → Validate → Transform → Validate → Load → Verify
#
# ============================================================

In [3]:
# ============================================================
# 1.- Preparación: posicionarse en la raíz del proyecto
# ============================================================
from pathlib import Path
import os

# Si Jupyter abrió el directorio notebooks/, regresamos a la raíz PyEconGDL.
if Path.cwd().name == "notebooks" and (Path.cwd().parent / "pipeline.py").exists():
    os.chdir("..")

print("Directorio de trabajo:", Path.cwd())
assert Path("pipeline.py").exists(), "Abre Jupyter desde la raíz del proyecto PyEconGDL"

Directorio de trabajo: C:\Users\marie\Documents\PyEconGDL


In [4]:
# ============================================================
# 2.- Sesión Spark local y verificación de entorno Hadoop/Java
# ============================================================
import os
os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jre1.8.0_503"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] += r";C:\hadoop\bin"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master("local[*]")
         .appName("Clase11_Pipeline_Deuda_Bruta")
         .config("spark.driver.host", "127.0.0.1")
         .config("spark.driver.bindAddress", "127.0.0.1")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.sql.adaptive.enabled", "false")
         .getOrCreate())

jvm = spark._jvm.java.lang.System
print("Spark:", spark.version)
print("Java:", jvm.getProperty("java.version"))
print("Arquitectura:", jvm.getProperty("sun.arch.data.model"))
print("Hadoop nativo:", spark._jvm.org.apache.hadoop.util.NativeCodeLoader.isNativeCodeLoaded())

Spark: 3.5.9
Java: 1.8.0_503
Arquitectura: 64
Hadoop nativo: True


In [5]:
# ============================================================
# 3.- Reconocer entrada y funciones reutilizables
# ============================================================
from pipeline import INPUT_PATH, OUTPUT_PATH, REPORT_PATH, GRANULARIDAD, run_pipeline
from src.ingestion.extract import extract
from src.quality.validate import validate_input, validate_output
from src.transformations.enrich import transform

df = extract(spark, INPUT_PATH)
df.printSchema()
print("Granularidad esperada:", GRANULARIDAD)
print("Registros de entrada:", df.count())
df.show(5, truncate=False)

[EXTRACT] Leyendo data/raw/archivo_deuda_bruta.parquet
root
 |-- CICLO: integer (nullable = true)
 |-- MES: string (nullable = true)
 |-- MONTO: long (nullable = true)

Granularidad esperada: mensual (una fila por CICLO + MES)
Registros de entrada: 186
+-----+-------+----------+
|CICLO|MES    |MONTO     |
+-----+-------+----------+
|2011 |Enero  |4430293041|
|2011 |Febrero|4503709153|
|2011 |Marzo  |4534934949|
|2011 |Abril  |4546027059|
|2011 |Mayo   |4585947148|
+-----+-------+----------+
only showing top 5 rows



In [6]:
# ============================================================
# 4.- Observar validación y transformación antes de ejecutar todo
# ============================================================
# Validación de entrada: devuelve el conteo de filas si cumple las reglas.
filas_entrada = validate_input(df)
print("Entrada validada:", filas_entrada)

# Transformación: construye PERIODO y aplica comparaciones intermensuales.
enriquecido = transform(df)
filas_salida = validate_output(enriquecido, filas_entrada)
print("Salida validada:", filas_salida)

enriquecido.orderBy("PERIODO").show(8, truncate=False)

[VALIDATE] Revisando entrada
Entrada validada: 186
[TRANSFORM] Construyendo PERIODO y comparando periodos mensuales
[VALIDATE] Revisando salida
Salida validada: 186
+----------+-----+-------+-------+----------+----------------------+---------------+-------------+-------------------+-------------------+----------------------+
|PERIODO   |CICLO|MES    |MES_NUM|MONTO     |MONTO_PERIODO_ANTERIOR|VARIACION_MONTO|VARIACION_PCT|MONTO_ANIO_ANTERIOR|VARIACION_ANUAL_PCT|PROMEDIO_MOVIL_3_MONTO|
+----------+-----+-------+-------+----------+----------------------+---------------+-------------+-------------------+-------------------+----------------------+
|2011-01-01|2011 |Enero  |1      |4430293041|NULL                  |NULL           |NULL         |NULL               |NULL               |4.430293041E9         |
|2011-02-01|2011 |Febrero|2      |4503709153|4430293041            |73416112       |1.6571       |NULL               |NULL               |4.467001097E9         |
|2011-03-01|2011 |Marzo  

In [7]:
# ============================================================
# 5.- Ejecutar el pipeline completo con el orquestador
# ============================================================
# run_pipeline coordina: extract -> validate -> transform -> validate -> load -> verify_load.
resultado = run_pipeline(spark, INPUT_PATH, OUTPUT_PATH, REPORT_PATH)
print("Salida Parquet creada:", Path(OUTPUT_PATH).exists())

=== INICIO DEL PIPELINE ===
[EXTRACT] Leyendo data/raw/archivo_deuda_bruta.parquet
[VALIDATE] Revisando entrada
[VALIDATE] Entrada correcta: 186 registros
[TRANSFORM] Construyendo PERIODO y comparando periodos mensuales
[VALIDATE] Revisando salida
[VALIDATE] Salida correcta: 186 registros
[LOAD] Guardando resultado Parquet
[LOAD] Verificando que la salida puede volver a leerse
[OK] Resultado: data/processed/deuda_bruta_enriquecida.parquet
=== FIN DEL PIPELINE ===
Salida Parquet creada: True


In [8]:
# ============================================================
# 6.- Verificar evidencia JSON y relectura del Parquet guardado
# ============================================================
import json

reporte = json.loads(Path(REPORT_PATH).read_text(encoding="utf-8"))
print("Reporte de ejecución:")
print(json.dumps(reporte, ensure_ascii=False, indent=2))

guardado = spark.read.parquet(OUTPUT_PATH)
print("\nFilas guardadas en disco:", guardado.count())
print("Columnas finales:", guardado.columns)

Reporte de ejecución:
{
  "fecha_utc": "2026-11-14T20:48:23.515890+00:00",
  "estado": "OK",
  "registros_entrada": 186,
  "registros_salida": 186,
  "granularidad": "mensual (una fila por CICLO + MES)",
  "ruta_salida": "data/processed/deuda_bruta_enriquecida.parquet",
  "periodo_inicial": "2011-01-01",
  "periodo_final": "2026-06-01",
  "error": null
}

Filas guardadas en disco: 186
Columnas finales: ['PERIODO', 'CICLO', 'MES', 'MES_NUM', 'MONTO', 'MONTO_PERIODO_ANTERIOR', 'VARIACION_MONTO', 'VARIACION_PCT', 'MONTO_ANIO_ANTERIOR', 'VARIACION_ANUAL_PCT', 'PROMEDIO_MOVIL_3_MONTO']


In [9]:
# ============================================================
# 7.- Cierre limpio de la sesión de Spark
# ============================================================
spark.stop()
print("Sesión de Spark cerrada correctamente.")

Sesión de Spark cerrada correctamente.
